# Supplemental Figure 1 — All of Us vs GTEx expression comparison


This notebook reproduces **Supplemental Figure 1**, comparing *All of Us* bulk
RNA-seq to GTEx whole blood across gene types: coding-gene TPM correlation and
cohort-specific detection, GO enrichment of cohort-specific genes, transcription-
factor expression, and lncRNA expression. Additional per-gene-type comparisons
are reproduced in the `extra/` sections at the end.

`00_setup.R` provides memoized builders (`gene_annotations()`, `merged_tpms()`,
`tf_list()`, `go_enrichment()`) cached to `SuppFig1/data/`.


## How to run

* **Kernel:** R (IRkernel). **Working directory:** the repository root (the folder that contains `SuppFig1/`).
* Run the cells top to bottom. Each panel caches its ggplot object to `SuppFig1/objects/`, and the assembly cell reads those back.
* Re-running a single panel cell rebuilds just that panel; re-run the assembly cell to recompose.


### Inputs

> **Controlled-access.** `aou_tpms` (and `go_annotations`) derive from *All of Us*
> controlled-access data. The GTEx TPMs, GENCODE GTF and TF list are public and
> self-stage from their `https://` URIs.

Inputs are manifest-driven (`SuppFig1/inputs.tsv`); builders cache to
`SuppFig1/data/` **by name**, so clear it after changing an input.

> **Results differ from the pre-cleanup script.** Panel C's TF statistics are
> logged once, not twice. See `SuppFig1/NOTES.md`.


## Setup

`00_setup.R` is the shared foundation every step builds on: it loads the libraries, sets the **Inter** font with a matched output **DPI**, defines the `theme_set`/`SZ_*` design system and the env-overridable paths, and brings in the manifest loaders and the folded-in analysis / plot functions plus the memoized derived-data builders — so the figure is self-contained.


In [ ]:
## Locate the figure directory. Terra / Linux is case-sensitive, so match
## either "SuppFig1" or "suppfig1" and point the pipeline's env-overridable
## paths at it before sourcing the shared setup.
fig_dir <- list.files(".", pattern = "^(SuppFig1|suppfig1)$", include.dirs = TRUE)
stopifnot("Run this notebook from the repository root (the folder holding SuppFig1/)." =
          length(fig_dir) == 1)
Sys.setenv(SUPPFIG1_MANIFEST = file.path(fig_dir, "inputs.tsv"),
           SUPPFIG1_OBJ = file.path(fig_dir, "objects"),
           SUPPFIG1_DATA = file.path(fig_dir, "data"))
source(file.path(fig_dir, "00_setup.R"))

## Inline rendering. The canvas resolution is matched to DPI because 00_setup.R
## pins showtext to it (showtext_opts(dpi = DPI)); if the two disagree, every
## label renders at the wrong size. Panels display at a readable default, and
## the assemble cell overrides these with the composed figure's true aspect.
options(repr.plot.width = 7.5, repr.plot.height = 5, repr.plot.res = DPI)
cat("Setup loaded from", fig_dir, "\n")


### Panels

Each panel below **loads only the inputs it needs** (staging them on demand), does its wrangling, builds one ggplot object, caches it to `SuppFig1/objects/`, and renders it inline.


### Panel A — protein-coding AoU-vs-GTEx TPM correlation (hex) + detection counts

Protein-coding genes detected in either cohort: AoU-vs-GTEx log2(TPM+1) correlation (hex) beside counts of genes detected in only one cohort. Sub-plots are tagged A and B.

*Inputs: aou_tpms, gtex_tpms, gencode_gtf*


In [ ]:
protein_coding_genes <- merged_tpms() %>%
  filter(gene_type == "protein_coding") %>%
  filter(mean_TPM_AoU > 1 | mean_TPM_GTEx > 1)

protein_coding_comparison <- protein_coding_genes %>%
  plot_gene_comparison(gene_type = "coding", hex = TRUE)

saveRDS(protein_coding_comparison, file.path(OBJ_DIR, "A.rds"))
message("Panel A built.")

protein_coding_comparison


### Panel B — GO enrichment of cohort-specific genes (AoU vs GTEx)

Top GO terms (by adjusted p) enriched among genes specific to each cohort, faceted AoU vs GTEx, with a -log10(FDR)=1.3 reference line. Tagged C.

*Inputs: aou_tpms, gtex_tpms, gencode_gtf (via go_enrichment)*


In [ ]:
AoU_GTEx_enrichment_plot <- go_enrichment() %>%
  # Original: arrange(Adjusted.P.value) %>% filter(row_number() < 10) keeps
  # rows 1-9, i.e. the top 9 terms per cohort (not 10).
  format_go_plot_data(top_n = 9) %>%
  ggplot(aes(x = neglog10_p, y = reorder(Term, neglog10_p))) +
  geom_col() +
  facet_wrap(~type, scales = "free_y") +
  scale_y_discrete(labels = function(x) str_wrap(x, width = 20)) +
  xlab("-log10(FDR)") + ylab("") +
  geom_vline(xintercept = 1.3) +
  theme(panel.spacing = unit(1, "cm"), strip.text = element_text(size = 20))

saveRDS(AoU_GTEx_enrichment_plot, file.path(OBJ_DIR, "B.rds"))
message("Panel B built.")

AoU_GTEx_enrichment_plot


### Panel C — TF expression boxplot (AoU vs GTEx), log2(TPM+1) once

Transcription-factor expression, AoU vs GTEx, as log2(TPM+1) boxplots, with an lm and a paired t-test reported in the run log. The TPMs are logged exactly once (the original logged twice). Tagged D.

*Inputs: aou_tpms, gtex_tpms, gencode_gtf, tf_list*


In [ ]:
TF_genes <- tf_expression_table(merged_tpms(), tf_list())

# Reported statistics (log exactly once, matching the axis label).
tf_lm <- TF_genes %>%
  do(broom::tidy(lm(log2(mean_TPM_GTEx + 1) ~ log2(mean_TPM_AoU + 1), data = .)))
tf_ttest <- t.test(log2(TF_genes$mean_TPM_AoU + 1),
                   log2(TF_genes$mean_TPM_GTEx + 1), paired = TRUE)
message("Panel C TF lm:"); print(tf_lm)
message("Panel C TF paired t-test p = ", signif(tf_ttest$p.value, 3))

TF_gene_comparison <- TF_genes %>%
  select(Description, contains("TPM")) %>%
  pivot_longer(!Description) %>%
  mutate(name = str_remove(name, "mean_TPM_")) %>%
  ggplot(aes(x = name, y = log2(value + 1), fill = name)) +
  geom_boxplot(show.legend = FALSE) +
  ylab("TF Expression \n log2(TPM + 1)") +
  scale_fill_carto_d(palette = "Safe")

saveRDS(TF_gene_comparison, file.path(OBJ_DIR, "C.rds"))
message("Panel C built.")

TF_gene_comparison


### Panel D — lncRNA AoU-vs-GTEx TPM correlation (hex) + detection counts

lncRNA AoU-vs-GTEx log2(TPM+1) correlation (hex) beside cohort-specific detection counts. Sub-plots are tagged E and F.

*Inputs: aou_tpms, gtex_tpms, gencode_gtf*


In [ ]:
lncRNA_comparison <- merged_tpms() %>%
  filter(gene_type == "lncRNA") %>%
  filter(mean_TPM_AoU > 0.1 | mean_TPM_GTEx > 0.1) %>%
  plot_gene_comparison(gene_type = "lncRNA", hex = TRUE)

saveRDS(lncRNA_comparison, file.path(OBJ_DIR, "D.rds"))
message("Panel D built.")

lncRNA_comparison


## Assemble the figure

Loads the cached panel objects from `SuppFig1/objects/` and composes them with patchwork into the final figure. Panel **tagging happens here and only here**. Writes the PDF plus a PNG preview, and renders the assembled figure inline.


In [ ]:
P <- setNames(lapply(LETTERS[1:4],
                     function(x) readRDS(file.path(OBJ_DIR, paste0(x, ".rds")))),
              LETTERS[1:4])

top_row    <- P$A | (P$B + theme(plot.margin = margin(l = 2, unit = "cm")))
bottom_row <- P$C | P$D

SuppFig1 <- top_row / bottom_row +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(size = SZ_TAG, face = "bold", family = FONT))

ggsave(file.path(OUTDIR, "SupplementalFig1.pdf"), SuppFig1,
       width = 35, height = 23, dpi = DPI, limitsize = FALSE)
showtext_opts(dpi = 100)
ggsave(file.path(OUTDIR, "SupplementalFig1.png"), SuppFig1,
       width = 35, height = 23, dpi = 100, limitsize = FALSE)
showtext_opts(dpi = DPI)

message("Assembled: ", file.path(OUTDIR, "SupplementalFig1.pdf"), " and .png")

options(repr.plot.width = 12.0, repr.plot.height = 7.9)
SuppFig1


## **Extra — snRNA / miRNA comparisons.** Retained per-gene-type AoU-vs-GTEx comparisons; not part of the assembled figure.


In [ ]:
merged <- merged_tpms()

for (gt in c("snRNA", "miRNA")) {
  p <- merged %>%
    filter(gene_type == gt) %>%
    filter(mean_TPM_AoU > 0.1 | mean_TPM_GTEx > 0.1) %>%
    plot_gene_comparison(gene_type = gt, hex = FALSE)
  ggsave(file.path(OUTDIR, paste0("SuppFig1_extra_", gt, ".pdf")),
         p, width = 10, height = 5, dpi = DPI)
  message("wrote SuppFig1_extra_", gt, ".pdf")
}


## **Extra — rRNA expression histogram.**


In [ ]:
rRNA_hist <- merged_tpms() %>%
  filter(gene_type == "rRNA") %>%
  select(Description, mean_TPM_AoU, mean_TPM_GTEx) %>%
  pivot_longer(!Description) %>%
  mutate(name = str_remove(name, "mean_TPM_")) %>%
  ggplot(aes(x = value, fill = name)) +
  geom_histogram(position = "identity", alpha = 0.7) +
  xlab("Mean rRNA TPM") + ylab("Number genes") +
  scale_fill_carto_d(palette = "Safe")

ggsave(file.path(OUTDIR, "SuppFig1_extra_rRNA.pdf"), rRNA_hist,
       width = 7, height = 5, dpi = DPI)
message("wrote SuppFig1_extra_rRNA.pdf")


## **Extra — hemoglobin-gene scatter.**


In [ ]:
hb_plot <- merged_tpms() %>%
  filter(str_detect(Description, "^HB") & gene_type == "protein_coding") %>%
  ggplot(aes(x = log2(mean_TPM_AoU), y = log2(mean_TPM_GTEx))) +
  geom_point(size = 6, shape = 21, fill = "black", alpha = 0.6) +
  geom_abline(slope = 1, color = "red") +
  ylim(-5, 20) + xlim(-5, 20) +
  xlab("log2(Mean AoU TPM)") + ylab("log2(Mean GTEx TPM)") +
  ggrepel::geom_text_repel(aes(label = Description))

ggsave(file.path(OUTDIR, "SuppFig1_extra_hemoglobin.pdf"), hb_plot,
       width = 7, height = 7, dpi = DPI)
message("wrote SuppFig1_extra_hemoglobin.pdf")


## **Extra — genes detected per gene type.**


In [ ]:
small_rnas_list <- c("misc_RNA", "sRNA", "vault_RNA", "scaRNA", "rRNA", "ribozyme")

# AoU per-gene means over ALL AoU-quantified genes (matches the original
# AoU_mean_TPMs_v45). Deliberately NOT merged_tpms(), which inner-joins AoU to
# GTEx and would narrow the gene universe to the AoU-GTEx intersection.
aou_means <- as_tibble(data.table::fread(input_path("aou_tpms"), skip = 2)) %>%
  compute_mean_TPM(gene_annotations())

gene_detection_summary <- aou_means %>%
  filter(mean_TPM > 0.1) %>%
  dplyr::count(gene_type) %>%
  mutate(gene_type = case_when(gene_type == "protein_coding" ~ "Coding", TRUE ~ gene_type)) %>%
  filter(!str_detect(gene_type, "IG|TR|pseudo|artifact|Mt|TEC|rRNA"))

other_small_rnas <- gene_detection_summary %>%
  filter(gene_type %in% small_rnas_list) %>%
  summarize(gene_type = "other small RNAs", n = sum(n))

detection_plot <- gene_detection_summary %>%
  bind_rows(other_small_rnas) %>%
  filter(!gene_type %in% small_rnas_list) %>%
  ggplot(aes(x = gene_type, y = n, fill = gene_type, color = gene_type)) +
  geom_col(show.legend = FALSE, alpha = 0.7) +
  ylab("Number detected \n (TPM > 0.1)") +
  scale_y_log10() +
  scale_fill_carto_d(palette = "Safe") + scale_color_carto_d(palette = "Safe") +
  geom_text(aes(label = n), vjust = -0.25, color = "black") +
  xlab("Type of gene")

ggsave(file.path(OUTDIR, "SuppFig1_extra_gene_detection.pdf"), detection_plot,
       width = 12, height = 7, dpi = DPI)
message("wrote SuppFig1_extra_gene_detection.pdf")
